# 3D U-Net + FADC-Encoder V2.1 + Attention Diversity Aux Loss

**What this notebook tests.** Whether adding an explicit batch-diversity aux loss on `c_att` and `f_att`, combined with a sharper k_att temperature schedule, breaks the c_att/f_att collapse observed on Encoder v2.1 s=42 (ep10 - ep30 diagnostics showed std < 0.005 at every layer past enc1, and enc4.c2 c_att std actively decaying from 0.0004 to 0.0002).

**Two changes vs plain v2.1 (branch `feature/attention-diversity-loss`, commit `eb4f6bd`):**

1. **Attention diversity aux loss** on every `OmniAttention3DSpatial` module.
   - Hinge penalty: `clamp(target_std - batch_std_over_channels, min=0)`, mean over layers.
   - Total training loss: `seg_loss + lambda * aux_loss`.
   - Once a layer's batch-std reaches the target, that layer contributes 0.
   - Defaults: lambda = 0.02, target = 0.03 (vs current dead-layer values of 0.0004 - 0.002).

2. **Sharper k_att anneal.**
   - `k_att_temp_end`: 0.8 -> 0.5 (spatial branch commits harder to per-voxel dilation).
   - `k_att_anneal_epochs`: full-100 -> int(0.6 * 100) = 60. Final 40 epochs run at T = 0.5 instead of a still-annealing softmax.

**Reference numbers to beat:**

| Reference | Best Val Dice | Notes |
|---|---|---|
| v2 fixed Encoder s=42 ep70 | 0.6267 | plateau v2.1 was trying to break |
| Encoder v2.1 s=42 ep30 | 0.5247 | current run, in progress |
| Baseline UNet3D 2ch | 0.6735 | uncontrolled seed |
| v1 Bottleneck s=42 | 0.6801 | project ceiling on Bottleneck placement |

**Decision gates for this run:**

| Best Dice | Interpretation | Follow-up |
|---|---|---|
| >= 0.69 | Diversity loss + sharper anneal actually unlock c_att/f_att into useful signal. | Queue seeds 123 + 999. Try Bottleneck placement. |
| 0.66 - 0.69 | Meaningful lift over Encoder v2.1's expected ep100 (~0.65). Aux loss helps. | One extra seed to disambiguate. |
| 0.63 - 0.66 | Matches Encoder v2.1 forecast band. Aux loss doesn't move the needle on Dice even if attention std grows. | Interpret as loss vs Dice signal — spatial c_att is the next fix to try. |
| < 0.63 | Aux loss is fighting segmentation loss. | Retry with lambda = 0.01 or disable aux and keep only the anneal change. |

**Diagnostic reads to watch (in addition to Dice):**

1. `aux_loss` in the per-epoch summary and `train_log.json`. Starts near `(target - baseline_std)` and should DECREASE as `c_att`/`f_att` std grows. Flat aux across many epochs = fix isn't landing.
2. Post-train c_att/f_att std at enc3/enc4 — did we cross the 0.005 gate at layers that were dead before?
3. k_att per-input-std — should be at least as strong as v2.1 s=42 ep30 (which had 3/8 layers >= 0.005). Watch for regressions.

**Setup:** FADC at enc1+enc2+enc3+enc4 only (no bn, no decoder). Patch 96 x 96 x 48. 100 epochs, batch 2, seed 42, cudnn deterministic, val_every = 10.

**GPU:** Kaggle T4 x 2. ~7 h expected wall time.

**IMPORTANT:** Download `best_model.pth`, `train_log.json`, `meta.json` at every val_every landing — `/kaggle/working` wipes on session close.

In [ ]:
# CONFIG
SEED = 42

DATA_ROOT              = "/kaggle/input/datasets/bharathvemurik/mama-mia-preprocessed-cache-2ch"
OUTPUT_DIR             = f"/kaggle/working/outputs/fadc_encoder_diversity_2ch_100ep_s{SEED}"
CODE_DIR               = "/kaggle/working/FADC-3D"
PREPROCESSED_CACHE_DIR = "/kaggle/input/datasets/bharathvemurik/mama-mia-preprocessed-cache-2ch"

EPOCHS       = 100
BATCH_SIZE   = 2
NUM_WORKERS  = 4
PATCH_SIZE   = [96, 96, 48]
WARMUP       = 5
VAL_EVERY    = 10

# k_att schedule (sharper than v2.1 defaults on this branch)
K_ATT_TEMP_START    = 4.0
K_ATT_TEMP_END      = 0.5      # was 0.8 on v2.1
K_ATT_ANNEAL_EPOCHS = 60       # was 100 on v2.1 — hold at t_end for last 40 epochs

# Attention diversity aux loss
ATTN_DIVERSITY_WEIGHT = 0.02   # lambda; 0.0 disables aux entirely
ATTN_DIVERSITY_TARGET = 0.03   # target batch-std for c_att / f_att

RESUME_FROM  = ""
GIT_BRANCH   = "feature/attention-diversity-loss"

print(f"SEED                    : {SEED}")
print(f"OUTPUT_DIR              : {OUTPUT_DIR}")
print(f"BRANCH                  : {GIT_BRANCH}")
print(f"k_att temp              : {K_ATT_TEMP_START} -> {K_ATT_TEMP_END} over {K_ATT_ANNEAL_EPOCHS} ep")
print(f"attn_diversity_weight   : {ATTN_DIVERSITY_WEIGHT}")
print(f"attn_diversity_target   : {ATTN_DIVERSITY_TARGET}")
print(f"val_every               : {VAL_EVERY}")

In [ ]:
# 1. INSTALL DEPENDENCIES
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "monai",
                "--upgrade-strategy", "only-if-needed", "-q"], check=True)

import torch
print(f"PyTorch        : {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    p = torch.cuda.get_device_properties(0)
    print(f"VRAM           : {p.total_memory / 1e9:.1f} GB")

In [ ]:
# 2. CLONE / UPDATE ATTENTION-DIVERSITY BRANCH
import os, sys
if os.path.exists(CODE_DIR):
    print(f"Repo exists — fetching + checkout {GIT_BRANCH} ...")
    os.system(f"git -C {CODE_DIR} fetch --all")
    os.system(f"git -C {CODE_DIR} checkout {GIT_BRANCH}")
    os.system(f"git -C {CODE_DIR} pull")
else:
    os.system(f"git clone -b {GIT_BRANCH} https://github.com/Vemuri-BK/FADC-3D.git {CODE_DIR}")
sys.path.insert(0, CODE_DIR)

for p in ["fadc_3d_v2/omni_attention_3d_spatial.py",
         "models/unet_3d_fadc_v2.py",
         "training/train_centralized_v2.py"]:
    assert os.path.exists(os.path.join(CODE_DIR, p)), f"missing: {p}"
print("Modules present.")

In [ ]:
# 3. PULL VERIFIER — abort if attention-diversity code did not land
# Guards against wasting 7h if git pull failed. Expected HEAD: eb4f6bd or later.
import inspect, sys, subprocess
sys.path.insert(0, CODE_DIR)
from fadc_3d_v2.omni_attention_3d_spatial import OmniAttention3DSpatial
from training import train_centralized_v2 as train_mod

# Print current branch tip
r = subprocess.run(["git", "-C", CODE_DIR, "log", "-1", "--oneline"],
                   capture_output=True, text=True)
print(f"HEAD  : {r.stdout.strip()}")

# --- V2 baseline checks (must still hold on this branch) ---
ch_src = inspect.getsource(OmniAttention3DSpatial.get_channel_attention)
fi_src = inspect.getsource(OmniAttention3DSpatial.get_filter_attention)
ka_src = inspect.getsource(OmniAttention3DSpatial.get_kernel_attention_spatial)
assert 'self.temperature' not in ch_src, 'v2 temperature-scope fix missing (get_channel_attention)'
assert 'self.temperature' not in fi_src, 'v2 temperature-scope fix missing (get_filter_attention)'
assert 'self.temperature' in ka_src,     'k_att lost its temperature term'
print('  [OK] v2 temperature-scope fix intact')

# --- V2.1 avg+max concat + always-on filter_fc ---
init_src = inspect.getsource(OmniAttention3DSpatial.__init__)
fwd_src  = inspect.getsource(OmniAttention3DSpatial.forward)
assert 'AdaptiveMaxPool3d' in init_src, 'v2.1 max pool NOT in __init__'
assert 'torch.cat' in fwd_src,          'v2.1 avg+max concat NOT in forward'
assert 'if in_planes == groups' not in init_src, 'filter_fc skip still present'
print('  [OK] v2.1 avg + max concat + always-on filter_fc')

# --- NEW on this branch: attention diversity aux loss helpers exist ---
assert hasattr(train_mod, 'register_attention_hooks'), \
    'register_attention_hooks missing — attention-diversity branch not pulled'
assert hasattr(train_mod, 'attention_diversity_loss'), \
    'attention_diversity_loss missing — attention-diversity branch not pulled'
assert hasattr(train_mod, '_AttnStore'), \
    '_AttnStore missing — attention-diversity branch not pulled'
print('  [OK] attention diversity helpers present in training module')

# --- CLI args landed with correct defaults ---
help_out = subprocess.run(
    [sys.executable, os.path.join(CODE_DIR, 'training', 'train_centralized_v2.py'), '--help'],
    capture_output=True, text=True).stdout
assert '--attn_diversity_weight' in help_out, '--attn_diversity_weight CLI missing'
assert '--attn_diversity_target' in help_out, '--attn_diversity_target CLI missing'
print('  [OK] --attn_diversity_weight and --attn_diversity_target are wired')

# --- Sharper anneal defaults present in help text ---
assert '0.5' in help_out, '--k_att_temp_end default 0.5 not visible in help'
assert '60%' in help_out or '0.6' in help_out, '60% anneal-window default not in help text'
print('  [OK] sharper k_att anneal defaults documented in --help')

print()
print('Verification passed — attention-diversity branch pulled correctly.')

In [ ]:
# 4. ARCH SMOKE — v2.1 encoder + all 8 FADC blocks + diversity aux loss end-to-end
import torch
from models.unet_3d_fadc_v2 import UNet3DFADC_V2
from fadc_3d_v2.adaptive_dilated_conv_3d_v2 import AdaptiveDilatedConv3DV2
from fadc_3d_v2.omni_attention_3d_spatial import OmniAttention3DSpatial
from training.train_centralized_v2 import register_attention_hooks, attention_diversity_loss

device = torch.device('cuda')
model = UNet3DFADC_V2(in_channels=2, out_channels=2, base_filters=32,
                       fadc_placement='encoder').to(device).train()
n_params = sum(p.numel() for p in model.parameters())
print(f"Params           : {n_params:,}")

# v2.1 shape signature check
for name, m in model.named_modules():
    if isinstance(m, OmniAttention3DSpatial):
        in_planes = m.channel_fc.out_channels
        assert m.fc.in_channels == 2 * in_planes, f"v2.1 avg+max concat missing at {name}"
        assert m.filter_fc is not None, f"filter_fc missing at {name}"
print('  [OK] v2.1 fc.in_channels == 2 * in_planes everywhere')

# Register aux hooks
store, handles = register_attention_hooks(model)
assert len(handles) == 8, f"expected 8 FADC layers on Encoder, got {len(handles)}"
print(f'  [OK] {len(handles)} attention hooks registered')

# Forward with B=2 so batch-std is meaningful
x = torch.randn(2, 2, *PATCH_SIZE, device=device)
target = torch.zeros(2, *PATCH_SIZE, device=device, dtype=torch.long)

store.clear()
store.enabled = True
y = model(x)
store.enabled = False
print(f"forward OK. output shape: {tuple(y.shape)}")
print(f"store captured: c_atts={len(store.c_atts)} f_atts={len(store.f_atts)}  (expect 8 each)")
assert len(store.c_atts) == 8 and len(store.f_atts) == 8

aux = attention_diversity_loss(store, target_std=ATTN_DIVERSITY_TARGET, device=device)
print(f"aux_loss = {aux.item():.6f}   grad_fn = {aux.grad_fn}")
assert aux.grad_fn is not None, 'aux_loss lost its grad_fn — backprop will not flow'

# Confirm backward reaches deepest c_att / f_att weights
seg_loss = torch.nn.functional.cross_entropy(y, target)
total = seg_loss + ATTN_DIVERSITY_WEIGHT * aux

deep_layer = None
for name, mod in model.named_modules():
    if name.endswith('enc4.conv.conv2.omni_att'):
        deep_layer = mod; break
assert deep_layer is not None

w0 = deep_layer.channel_fc.weight.detach().clone()
f0 = deep_layer.filter_fc.weight.detach().clone()
model.zero_grad()
total.backward()
assert deep_layer.channel_fc.weight.grad is not None, 'no grad on channel_fc.weight at enc4.c2'
assert deep_layer.filter_fc.weight.grad  is not None, 'no grad on filter_fc.weight at enc4.c2'
print('  [OK] backward reached channel_fc and filter_fc at enc4.conv.conv2')

for h in handles: h.remove()
del model, x, y, aux, seg_loss, total
torch.cuda.empty_cache()
print('\nEncoder + aux-loss smoke test PASSED.')

In [ ]:
# 5. CACHE SANITY
import os, numpy as np
from pathlib import Path

cache_path = Path(PREPROCESSED_CACHE_DIR)
assert cache_path.exists(), f"Cache not found: {cache_path}"
train_npzs = sorted((cache_path / "train").glob("*.npz"))
val_npzs   = sorted((cache_path / "val").glob("*.npz"))
print(f"Train : {len(train_npzs)} | Val: {len(val_npzs)}")
for p in [train_npzs[0], train_npzs[-1], val_npzs[0]]:
    d = np.load(p)
    print(f"  {p.name}  image={d['image'].shape}  label={d['label'].shape}")
    assert d['image'].shape[0] == 2, f"NOT 2-channel: {p.name}"
print("Cache OK.")

In [ ]:
# 6. LAUNCH TRAINING
import os, subprocess, sys
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_script = os.path.join(CODE_DIR, "training", "train_centralized_v2.py")

cmd = [
    sys.executable, "-u", train_script,
    "--model",          "unet3d_fadc_encoder_v2",
    "--data_root",      DATA_ROOT,
    "--output_dir",     OUTPUT_DIR,
    "--epochs",         str(EPOCHS),
    "--batch_size",     str(BATCH_SIZE),
    "--num_workers",    str(NUM_WORKERS),
    "--patch_size",     str(PATCH_SIZE[0]), str(PATCH_SIZE[1]), str(PATCH_SIZE[2]),
    "--warmup_epochs",  str(WARMUP),
    "--val_every",      str(VAL_EVERY),
    "--seed",           str(SEED),
    "--k_att_temp_start",       str(K_ATT_TEMP_START),
    "--k_att_temp_end",         str(K_ATT_TEMP_END),
    "--k_att_anneal_epochs",    str(K_ATT_ANNEAL_EPOCHS),
    "--attn_diversity_weight",  str(ATTN_DIVERSITY_WEIGHT),
    "--attn_diversity_target",  str(ATTN_DIVERSITY_TARGET),
]
if RESUME_FROM:
    cmd += ["--resume", RESUME_FROM]
if PREPROCESSED_CACHE_DIR:
    cmd += ["--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR]

print("Command:\n  " + " ".join(cmd))
print("=" * 60, flush=True)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = process.stdout.read(512)
    if not chunk:
        break
    sys.stdout.write(chunk.decode("utf-8", errors="replace"))
    sys.stdout.flush()
process.wait()
print(f"\nExit code: {process.returncode}")

In [ ]:
# 7. TRAINING CURVES — Loss, Val Dice, Aux loss, k_att temperature
import json, os
import matplotlib.pyplot as plt

BASELINE_DICE       = 0.6735
V2_FIXED_EP70       = 0.6267
V21_EP30            = 0.5247    # Encoder v2.1 s=42 ep30 (this branch's direct comparator)
BOTTLENECK_V1_DICE  = 0.6801

log_path = os.path.join(OUTPUT_DIR, "train_log.json")
if not os.path.exists(log_path):
    print("No training log yet.")
else:
    with open(log_path) as f:
        log = json.load(f)
    epochs     = [e["epoch"] for e in log]
    losses     = [e["loss"]  for e in log]
    aux_losses = [e.get("aux_loss", 0.0) for e in log]
    temps      = [e.get("k_att_temperature", float("nan")) for e in log]
    val_epochs = [e["epoch"]    for e in log if "val_dice" in e]
    val_dices  = [e["val_dice"] for e in log if "val_dice" in e]

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    ax = axes[0, 0]
    ax.plot(epochs, losses, color="steelblue")
    ax.set_title("Total Training Loss"); ax.set_xlabel("Epoch"); ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    ax.plot(val_epochs, val_dices, color="darkorange", marker="o", markersize=4, label="this run")
    ax.axhline(V21_EP30,          color="gold",   ls="-",  label=f"v2.1 s=42 ep30 ({V21_EP30:.4f})")
    ax.axhline(V2_FIXED_EP70,     color="brown",  ls="-",  label=f"v2 fixed ep70 ({V2_FIXED_EP70:.4f})")
    ax.axhline(BASELINE_DICE,     color="green",  ls="--", label=f"Baseline ({BASELINE_DICE:.4f})")
    ax.axhline(BOTTLENECK_V1_DICE, color="purple", ls=":",  label=f"v1 Bottleneck s=42 ({BOTTLENECK_V1_DICE:.4f})")
    if val_dices:
        ax.set_title(f"Val Dice  (best: {max(val_dices):.4f})")
    else:
        ax.set_title("Val Dice")
    ax.set_xlabel("Epoch"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    ax.plot(epochs, aux_losses, color="crimson")
    ax.axhline(0.0, color="gray", ls=":")
    ax.set_title("Attention Diversity Aux Loss (hinge; should DROP as std grows)")
    ax.set_xlabel("Epoch"); ax.set_ylabel("aux_loss"); ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    ax.plot(epochs, temps, color="darkred")
    ax.set_title(f"k_att T ({K_ATT_TEMP_START} -> {K_ATT_TEMP_END} over {K_ATT_ANNEAL_EPOCHS} ep)")
    ax.set_xlabel("Epoch"); ax.set_ylabel("T"); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150)
    plt.show()

    if val_dices:
        best = max(val_dices)
        print(f"Best Val Dice            : {best:.4f}")
        print(f"vs v2.1 s=42 ep30        : {best - V21_EP30:+.4f}   (direct comparator)")
        print(f"vs v2 fixed ep70         : {best - V2_FIXED_EP70:+.4f}")
        print(f"vs Baseline              : {best - BASELINE_DICE:+.4f}")
        print(f"vs v1 Bottleneck s=42    : {best - BOTTLENECK_V1_DICE:+.4f}")

    if aux_losses:
        print(f"\nAux loss: start {aux_losses[0]:.4f}  -> end {aux_losses[-1]:.4f}")
        print(f"(hinge target {ATTN_DIVERSITY_TARGET}; aux dropping = c_att/f_att batch-std growing)")

In [ ]:
# 8. POST-TRAIN DIAGNOSTIC — c_att / f_att std AND k_att per-input adaptation
# Repeats the diag_v2_attention.py structure on the best_model.pth landed here.
import os, sys, torch, numpy as np
sys.path.insert(0, CODE_DIR)
from models.unet_3d_fadc_v2 import UNet3DFADC_V2
from fadc_3d_v2.omni_attention_3d_spatial import OmniAttention3DSpatial

ckpt_path = os.path.join(OUTPUT_DIR, "best_model.pth")
if not os.path.exists(ckpt_path):
    print("No best_model.pth — run training first.")
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    cfg = ckpt['config']
    epoch = ckpt.get('epoch', 0)
    print(f"Loaded ckpt: epoch={epoch} (0-idx), best_dice={ckpt.get('best_dice'):.4f}")

    model = UNet3DFADC_V2(in_channels=cfg['model']['in_channels'],
                          out_channels=cfg['model']['out_channels'],
                          base_filters=cfg['model']['base_filters'],
                          fadc_placement='encoder').to(device).eval()
    model.load_state_dict(ckpt['model'])

    # Set the temperature the training loop would have set at that epoch
    import math
    def est_T(e, ae=K_ATT_ANNEAL_EPOCHS, ts=K_ATT_TEMP_START, te=K_ATT_TEMP_END):
        if ae <= 1: return te
        e = min(max(e, 0), ae - 1)
        c = 0.5 * (1.0 + math.cos(math.pi * e / (ae - 1)))
        return te + (ts - te) * c
    T_est = est_T(epoch)
    if hasattr(model, 'set_temperature'):
        model.set_temperature(T_est)
    print(f"Estimated k_att T at this epoch : {T_est:.3f}")

    # Capture c_att / f_att / k_att from every OmniAttention3DSpatial
    capture = {}
    handles = []
    for name, mod in model.named_modules():
        if isinstance(mod, OmniAttention3DSpatial):
            capture[name] = {'c': [], 'f': [], 'k': []}
            def mk(nm):
                def hook(_m, _i, out):
                    c_att, f_att, _s, k_att = out
                    capture[nm]['c'].append(c_att.detach().cpu())
                    capture[nm]['f'].append(f_att.detach().cpu())
                    capture[nm]['k'].append(k_att.detach().cpu())
                return hook
            handles.append(mod.register_forward_hook(mk(name)))

    # Feed 8 different random inputs
    torch.manual_seed(0)
    N = 8
    with torch.no_grad():
        for _ in range(N):
            x = torch.randn(1, 2, 32, 32, 16, device=device)
            _ = model(x)
    for h in handles: h.remove()

    print()
    print('=' * 108)
    print(f"{'Layer':<28} {'c_att mean':>11} {'c_att std':>11} "
          f"{'f_att mean':>11} {'f_att std':>11} "
          f"{'k spatial-std':>15} {'k per-input-std':>17}")
    print('-' * 108)
    for name in sorted(capture):
        c = torch.stack(capture[name]['c'], 0)
        f = torch.stack(capture[name]['f'], 0)
        k = torch.stack(capture[name]['k'], 0)
        c_mean = c.mean().item()
        c_std  = c.mean(dim=(2, 3, 4, 5)).std().item()
        f_mean = f.mean().item()
        f_std  = f.mean(dim=(2, 3, 4, 5)).std().item()
        k0 = k[:, 0, 0]  # branch 0 spatial map per input
        spatial_std = k0.flatten(1).std(dim=1).mean().item()
        per_input_std = k0.mean(dim=(1, 2, 3)).std().item()
        print(f"{name:<28} {c_mean:>11.4f} {c_std:>11.4f} "
              f"{f_mean:>11.4f} {f_std:>11.4f} "
              f"{spatial_std:>15.4f} {per_input_std:>17.4f}")
    print('=' * 108)

    # Reference numbers from v2.1 s=42 ep30 (this branch's comparator)
    print()
    print('Reference — v2.1 s=42 ep30 (WITHOUT aux loss, WITHOUT sharper anneal):')
    print('  enc4.c2 c_att std : 0.0002    <- watch: aux loss should push this above 0.005')
    print('  enc3.c1 c_att std : 0.0022    <- watch: was plateaued; aux loss should keep growing it')
    print('  enc4.c2 k_att per-input-std : 0.0174  <- should stay >= this level')
    print('  Per-input adaptation count  : 3/8 layers  <- aim: 4-6 / 8')

In [ ]:
# 9. DOWNLOAD LINKS
import os
from IPython.display import FileLink, display
for fname in ("best_model.pth", "latest_checkpoint.pth", "train_log.json", "meta.json", "training_curves.png"):
    p = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(p):
        print(fname); display(FileLink(p))
    else:
        print(f"(missing) {fname}")